In [2]:
import pandas as pd

In [3]:
input_paths = [
    "../Data/0_example.txt",
    "../Data/1_binary_landscapes.txt",
    "../Data/10_computable_moments.txt",
    "../Data/11_randomizing_paintings.txt",
    "../Data/110_oily_portraits.txt"
    ]

output_paths = [
        "../output/0_example.txt",
        "../output/1_binary_landscapes.txt",
        "../output/10_computable_moments.txt",
        "../output/11_randomizing_paintings.txt",
        "../output/110_oily_portraits.txt"
    ]


In [4]:
file_number = 2

input_path = input_paths[file_number]
output_path = output_paths[file_number]

with open(input_path, 'r') as file:
    lines = file.read().strip().split('\n')
lines = lines[1:]

In [5]:

parsed = []
index = 0
for line in lines:
    parts = line.strip().split()
    painting_type = parts[0]
    num_tags = int(parts[1])
    tags = set(parts[2:]) 
    parsed.append({
        "index" : index,
        "Type": painting_type,
        "Num_Tags": num_tags,
        "Tags": tags
    })
    index += 1
df = pd.DataFrame(parsed)
# df = df.sort_values(by='Type', ascending=False)
print(df)

     index Type  Num_Tags                                               Tags
0        0    P         7            {kg6, k1l, kmz1, k8b1, k17, kjb1, kw52}
1        1    L         7             {kgr, kc51, kzb1, k81, krn, k6c, kwk1}
2        2    P         8       {k001, k81, k652, kb6, kd7, k4q, kg11, k201}
3        3    L        10  {k21, kqp1, kms, k0c, kqz1, kg02, kwd1, k8b1, ...
4        4    L        12  {kpb, ksj1, k4k1, kv81, k892, krn, kz41, kk, k...
..     ...  ...       ...                                                ...
995    995    P        14  {kgp1, kbd1, kr1, kqq, kwx, kk, kwj1, k512, kv...
996    996    P         9   {kmn, krg1, khk, kb5, k9d, k1s, k8b1, k0w1, kv1}
997    997    L         8      {k452, k881, k3b2, k19, khh, k311, kpn1, k6c}
998    998    L        10  {kdk, k311, k3p1, kjj1, kvc1, k24, k42, kpl, k...
999    999    P         5                        {kqx, k4q, kpc, k031, k3j1}

[1000 rows x 4 columns]


In [6]:

df_p = df[df['Type'] == 'P'].copy()
df_l = df[df['Type'] == 'L'].copy()
merged = []
for i in range(0, len(df_p), 2):
    if i + 1 < len(df_p):
        idx1, idx2 = df_p.iloc[i]['index'], df_p.iloc[i+1]['index']
        tags1, tags2 = df_p.iloc[i]['Tags'], df_p.iloc[i+1]['Tags']
        tags3 = tags1.union(tags2)
        merged.append({
                "index": f"{idx1} {idx2}",
                "Type" : "P",
                "Tags": tags3,
                "Num_Tags" : len(tags3)
            })


In [7]:
merged_df = pd.DataFrame(merged)
df = pd.concat([merged_df, df_l], ignore_index=True)
print(df)

     index Type                                               Tags  Num_Tags
0      0 2    P  {k001, kg6, k81, k652, kb6, k1l, kd7, kmz1, k8...        15
1      6 8    P  {kc61, k502, kpz1, k3k, klf, kk6, k76, kr5, k1...        24
2    10 12    P  {kkb2, ks3, kk51, kw6, k0l1, k582, kh8, k3d1, ...        24
3    15 18    P  {kzg1, kl21, k6d1, k82, kg7, klb1, kd7, kr82, ...        24
4    20 26    P  {kmr, kj21, kg82, k3m1, k112, k9x, k8q, k0c, k...        27
..     ...  ...                                                ...       ...
745    989    L                     {kgk1, kb91, k462, k4m1, kjb2}         5
746    992    L  {kpr1, khc, kqs, kqp1, kp41, kk01, kpc, kk82, ...        11
747    994    L  {k1d2, k5p1, k372, kw6, k3q, kdg, k1b, k2c, kd...        13
748    997    L      {k452, k881, k3b2, k19, khh, k311, kpn1, k6c}         8
749    998    L  {kdk, k311, k3p1, kjj1, kvc1, k24, k42, kpl, k...        10

[750 rows x 4 columns]


In [ ]:
# Greedy algorithm
def greedy_reorder(df):
    used = set()
    order = []
    
    # Start with the row with max total tag overlap with others
    start = max(df.index, key=lambda i: sum(len(df.loc[i, 'Tags'] & df.loc[j, 'Tags']) for j in df.index if i != j))
    current = start
    used.add(current)
    order.append(current)

    while len(used) < len(df):
        next_index = max(
            (i for i in df.index if i not in used),
            key=lambda i: len(df.loc[current, 'Tags'] & df.loc[i, 'Tags']),
            default=None
        )
        if next_index is None:
            break
        used.add(next_index)
        order.append(next_index)
        current = next_index
    
    return df.loc[order].reset_index(drop=True)


chunk_size = 300
chunks = [df[i:i + chunk_size].reset_index(drop=True) for i in range(0, len(df), chunk_size)]

processed_chunks = []
for chunk in chunks:
    chunk = greedy_reorder(chunk)
    processed_chunks.append(chunk)

final_df = pd.concat(processed_chunks, ignore_index=True)



In [9]:
import winsound    

winsound.Beep(1440, 200)  

In [ ]:
output = final_df['index']  
len_output = len(output)

with open(output_path, "w") as f:
    f.write(str(len_output) + "\n")
    for line in output.values:
        f.write(str(line)+'\n')

NameError: name 'ordered_df' is not defined